In [1]:
from preprocessing import preprocess_data
from feature_engineering import run_feature_engineering
from make_model import train_and_evaluate
from feature_importance import analyze_feature_importance
from inference import predict_score
from verify_training_data import diagnose_training_data

### Preprocessing
Raw data from multiple rides with the earable is combined into a master `.csv` per sensors, e.g. `master_gyroscope.csv`. Folders with raw data are stored in the same directory as this file, where the last digit character of the folder name is the distraction score.

In [14]:
preprocess_data()

Starting data preprocessing...
Saved Labels: ./processed/y_labels.csv (12 rides labeled)
Saved Sensor Data: ./processed/master_temperature_sensor.csv (19825 total rows across all rides)
Saved Sensor Data: ./processed/master_optical_temperature_sensor.csv (24662 total rows across all rides)
Saved Sensor Data: ./processed/master_barometer.csv (19825 total rows across all rides)
Saved Sensor Data: ./processed/master_gyroscope.csv (316897 total rows across all rides)
Saved Sensor Data: ./processed/master_magnetometer.csv (316885 total rows across all rides)
Saved Sensor Data: ./processed/master_accelerometer.csv (316909 total rows across all rides)
Saved Sensor Data: ./processed/master_photoplethysmography.csv (158649 total rows across all rides)

Preprocessing complete, data is ready for tsfresh feature extraction


### Feature engineering using tsfresh
The master `.csv` is processed by tsfresh to extract features such as maximum value, standard deviation, median, etc. which can be processed by XGBoost. Exclude sensors by putting them in the `exclude_sensor` list.

In [15]:
# exclude_sensors = ["optical_temperature_sensor", "accelerometer", "gyroscope", "magnetometer", "barometer", "photoplethysmography", "temperature_sensor"]

# We suspect temperature gives a wrong indication due to the way we ordered the rides:
# The first ride was with high distraction score and the last ride with low distraction score. 
# So the temperature sensor is basically just learning to predict the order of the rides, which is not what we want.
exclude_sensors = ["optical_temperature_sensor", "temperature_sensor","magnetometer","accelerometer_1"]

run_feature_engineering(exclude_sensors=exclude_sensors)

Starting tsfresh feature engineering...
Extracting features for: barometer


Feature Extraction: 100%|██████████| 12/12 [00:00<00:00, 288.10it/s]

Extracting features for: photoplethysmography



Feature Extraction: 100%|██████████| 48/48 [00:00<00:00, 588.06it/s]


Extracting features for: gyroscope


Feature Extraction: 100%|██████████| 36/36 [00:00<00:00, 449.46it/s]


Extracting features for: accelerometer


Feature Extraction: 100%|██████████| 36/36 [00:00<00:00, 361.24it/s]


Merging data...
Total features kept for training: 110

Success, final dataset saved to: ./training_data/X_selected_features.csv


### Train and evaluate model using XGBoost

In [16]:
train_and_evaluate(custom_labels=False)

Dataset ready: 12 rides, 110 features per ride.
Evaluating model using Leave-One-Out Cross-Validation...

MODEL PERFORMANCE
Mean Absolute Error (MAE): 0.81
Root Mean Squared Error (RMSE): 0.91
The model's guesses are off by an average of 0.81 points on the 0-4 scale.

Training final production model on 100% of the data...
Trained model saved to: ./models/model.json


### Evaluate feature importance of model
XGBoost has built-in functions to show the importance per feature.

In [10]:
analyze_feature_importance()


Top 15 Most Important Features for Predicting Distraction:
----------------------------------------------------------------------
Rank  | Feature Name                                  | Importance Score
----------------------------------------------------------------------
1     | GYROSCOPE_Z__standard_deviation               | 0.3597
2     | GYROSCOPE_X__standard_deviation               | 0.2960
3     | BAROMETER_Pressure__sum_values                | 0.2373
4     | BAROMETER_Pressure__standard_deviation        | 0.0639
5     | BAROMETER_Pressure__minimum                   | 0.0186
6     | GYROSCOPE_Y__minimum                          | 0.0087
7     | GYROSCOPE_Y__standard_deviation               | 0.0083
8     | BAROMETER_Pressure__median                    | 0.0029
9     | GYROSCOPE_X__maximum                          | 0.0028
10    | PHOTOPLETHYSMOGRAPHY_RED__sum_values          | 0.0018
----------------------------------------------------------------------

Full importance ranking

### Run inference on a single ride
To run inference on a single ride (folder), give `new_ride_folder="..."` as an argument and the model will give a distraction score.

In [3]:
score = predict_score(new_ride_folder="runs/OpenWearable_Recording_2026-03-26T103015.953397_1")

Processing new ride: OpenWearable_Recording_2026-03-26T103015.953397_1
Calculated distraction score: 1.16 / 4.0


### Check scores for all the rides in the training data
Gives a list of rides used in training the model and their scores according to the model

In [12]:
diagnose_training_data()

Loading data and model for diagnostics...

Ride ID (Shortened)       | True Score | Predicted  | Error     
-----------------------------------------------------------------
2026-03-26T101341.966067  | 4.0        | 3.71       | 0.29      
2026-03-26T101935.303786  | 3.0        | 2.89       | 0.11      
2026-03-26T102512.493306  | 2.0        | 2.13       | 0.13      
2026-03-26T103015.953397  | 1.0        | 1.16       | 0.16      
2026-03-26T103513.282691  | 0.0        | 0.53       | 0.53      
2026-03-26T105712.346697  | 4.0        | 3.61       | 0.39      
2026-03-26T110257.381741  | 3.0        | 2.84       | 0.16      
2026-03-26T110809.275003  | 2.0        | 2.07       | 0.07      
2026-03-26T111245.688277  | 1.0        | 1.09       | 0.09      
2026-03-26T111650.158656  | 0.0        | 0.51       | 0.51      
2026-03-26T112950.696222  | 4.0        | 3.60       | 0.40      
2026-03-26T113555.625881  | 3.0        | 2.96       | 0.04      
----------------------------------------------